# Task 03 — Feature Engineering & Preprocessing Pipeline
**RabTech Academy | Artificial Intelligence & Machine Learning Internship**

## Objective
Build a reusable preprocessing pipeline for the provided customer-churn dataset. This notebook covers data inspection, missing-value handling, encoding, scaling, train/test splitting, correlation analysis, and feature importance.

**Dataset limitation:** The provided CSV has only 12 rows and 7 columns. Use it to demonstrate the workflow only. Its performance metrics are not reliable enough for real-world decisions.

## How to use this project
1. Keep `customer_churn_training.csv` in the same folder as this notebook.
2. Run the notebook from top to bottom using **Run All**.
3. Review the printed metrics and plots.

## Step 1 — Import required libraries

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)

## Step 2 — Load the CSV and inspect it

In [ ]:
DATA_PATH = Path("customer_churn_training.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Put customer_churn_training.csv in the same folder as this notebook.")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("\\nData types:")
display(df.dtypes.astype(str).to_frame("dtype"))
print("\\nSummary statistics:")
display(df.describe(include="all").T)

## Step 3 — Check data quality

In [ ]:
TARGET = "churned"
ID_COLUMN = "customer_id"

print("Missing values:")
display(df.isna().sum().to_frame("missing_count"))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Duplicate customer IDs:", int(df[ID_COLUMN].duplicated().sum()))
print("Target counts:")
display(df[TARGET].value_counts().sort_index().rename(index={0: "Not churned (0)", 1: "Churned (1)"}).to_frame("count"))

ax = df[TARGET].value_counts().sort_index().plot(kind="bar", rot=0)
ax.set_title("Target class distribution")
ax.set_xlabel("churned (0 = no, 1 = yes)")
ax.set_ylabel("Number of rows")
plt.tight_layout()
plt.show()

## Step 4 — Split the data BEFORE preprocessing

The target is `churned` (1 means churned; 0 means not churned). We exclude `customer_id` because it is an identifier rather than a useful behavioral feature.

We split first, then fit the imputers, scaler, encoder, and model using only training data. This is the key step to prevent data leakage.

In [ ]:
X = df.drop(columns=[TARGET, ID_COLUMN])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training churn rate:", round(y_train.mean(), 3))
print("Test churn rate:", round(y_test.mean(), 3))

## Step 5 — Separate numerical and categorical features

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

## Step 6 — Build the preprocessing pipeline

- Numerical features: median imputation followed by standard scaling.
- Categorical features: most-frequent imputation followed by one-hot encoding.
- `ColumnTransformer` applies each preprocessing path to the correct columns.
- `Pipeline` combines preprocessing and modeling so the transformations are fitted only on training data.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

pipeline.fit(X_train, y_train)
print("Preprocessing pipeline and Random Forest fitted successfully.")

## Step 7 — Compare with a simple non-ML baseline

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
model_pred = pipeline.predict(X_test)

def metric_row(name, actual, predicted):
    return {
        "model": name,
        "accuracy": accuracy_score(actual, predicted),
        "precision": precision_score(actual, predicted, zero_division=0),
        "recall": recall_score(actual, predicted, zero_division=0),
        "f1_score": f1_score(actual, predicted, zero_division=0)
    }

results = pd.DataFrame([
    metric_row("Majority-class baseline", y_test, baseline_pred),
    metric_row("Random Forest pipeline", y_test, model_pred)
])
display(results.round(3))

print("Confusion matrix (rows=actual, columns=predicted; labels 0 then 1):")
print(confusion_matrix(y_test, model_pred, labels=[0, 1]))
print(classification_report(y_test, model_pred, zero_division=0))
print("CAUTION: only 3 test rows; these scores are highly unstable.")

## Step 8 — Correlation analysis

In [ ]:
corr = X_train[numeric_features].corr(numeric_only=True)
display(corr.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.set_title("Correlation matrix — training numerical features")
fig.colorbar(im, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## Step 9 — Feature importance

In [ ]:
fitted_preprocessor = pipeline.named_steps["preprocessor"]
fitted_model = pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": fitted_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df.round(4))

plot_df = importance_df.sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df["feature"], plot_df["importance"])
ax.set_title("Random Forest feature importance")
ax.set_xlabel("Impurity-based importance")
plt.tight_layout()
plt.show()

## Step 10 — Predict on held-out rows

In [ ]:
preview = X_test.copy()
preview["actual_churned"] = y_test
preview["predicted_churned"] = pipeline.predict(X_test)
display(preview)

## Step 11 — Conclusion and limitations

This notebook demonstrates:
- Numerical missing-value imputation and scaling.
- Categorical missing-value imputation and one-hot encoding.
- A train/test split before fitting transformations.
- A reusable `ColumnTransformer` and `Pipeline`.
- Correlation analysis and tree-based feature importance.
- Comparison against a simple majority-class baseline.

**Limitations:** The CSV contains only 12 rows, so model performance and feature rankings are not reliable. Dataset collection method, consent, license, and exact churn definition were not supplied; confirm these with the data provider. Do not use this demonstration to make real customer decisions.